# 04 — Logistic Regression

TF-IDF is fitted only on the training set. The same saved split is used by the LSTM notebook for a fair comparison.

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

ROOT = Path.cwd()
DATA = ROOT / "data" / "processed" / "cleaned_news.csv"
SPLIT = ROOT / "data" / "processed" / "model_split.csv"
MODELS = ROOT / "models"; METRICS = ROOT / "reports" / "metrics"; FIGURES = ROOT / "reports" / "figures"
MODELS.mkdir(exist_ok=True); METRICS.mkdir(parents=True, exist_ok=True); FIGURES.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA)
split = pd.read_csv(SPLIT)
train_idx = split.loc[split.split == "train", "row_index"].to_numpy()
test_idx = split.loc[split.split == "test", "row_index"].to_numpy()
X_train, y_train = df.loc[train_idx, "clean_text"], df.loc[train_idx, "label"]
X_test, y_test = df.loc[test_idx, "clean_text"], df.loc[test_idx, "label"]

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english")
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LogisticRegression(C=1.0, max_iter=1000, solver="liblinear", random_state=42)
model.fit(X_train_tfidf, y_train)

y_prob = model.predict_proba(X_test_tfidf)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

metrics = {
    "model": "Logistic Regression",
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1_score": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_prob),
    "test_size": len(y_test),
}
print(metrics)
print(classification_report(y_test, y_pred, target_names=["Fake News", "Real News"], zero_division=0))

In [ ]:
joblib.dump(model, MODELS / "logistic_regression_model.joblib")
joblib.dump(vectorizer, MODELS / "tfidf_vectorizer.joblib")
pd.DataFrame([metrics]).to_csv(METRICS / "logistic_regression_metrics.csv", index=False)

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=["Fake", "Real"]).plot(ax=ax)
ax.set_title("Logistic Regression Confusion Matrix")
fig.tight_layout()
fig.savefig(FIGURES / "logistic_regression_confusion_matrix.png", dpi=150)
plt.show()
print("Artifacts saved to models/ and reports/")

In [ ]:
def predict_news_logistic(text: str):
    from src.preprocessing import preprocess_text
    cleaned = preprocess_text(text)
    features = vectorizer.transform([cleaned])
    probability = float(model.predict_proba(features)[0, 1])
    label = int(probability >= 0.5)
    return ("Real News" if label else "Fake News", probability)

predict_news_logistic("Government officials announced new economic reforms after a press conference.")